# 벡터: 신경망의 기초 - 벡터의 기초와 기하학적 의미

- Tutorial ID: `math-1-1`
- Tutorial: 벡터: 신경망의 기초
- Section ID: `math-1-1-1`
- Section: 벡터의 기초와 기하학적 의미


In [ ]:
# ============================================================
# 코드 읽는 법 — 벡터의 기초와 기하학적 의미
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 입력 데이터가 어떤 중간 변수들을 거쳐 최종 출력으로 변환되는지 shape 중심으로 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("벡터의 기초: 신경망을 위한 선형대수")
print("=" * 60)

In [ ]:
# 1. 벡터 생성과 기본 연산

In [ ]:
print("\n1. 벡터 기본 연산")
print("-" * 45)

# 벡터 생성
v = np.array([3.0, 4.0])
w = np.array([1.0, 2.0])

print(f"v = {v}")
print(f"w = {w}")
print(f"v + w = {v + w}")
print(f"2 * v = {2 * v}")
print(f"v - w = {v - w}")

In [ ]:
# 2. 내적 (Dot Product / Inner Product)

In [ ]:
print("\n2. 내적 (Dot Product)")
print("-" * 45)

dot_product = np.dot(v, w)  # 또는 v @ w
print(f"v · w = {dot_product}")
print(f"  = {v[0]}×{w[0]} + {v[1]}×{w[1]}")
print(f"  = {v[0]*w[0]} + {v[1]*w[1]} = {v[0]*w[0] + v[1]*w[1]}")

# 내적의 기하학적 의미: ||v|| ||w|| cos(θ)
norm_v = np.linalg.norm(v)
norm_w = np.linalg.norm(w)
cos_theta = dot_product / (norm_v * norm_w)
theta_rad = np.arccos(cos_theta)
theta_deg = np.degrees(theta_rad)

print(f"\n기하학적 해석:")
print(f"  ||v|| = √(3² + 4²) = {norm_v}")
print(f"  ||w|| = √(1² + 2²) = {norm_w:.4f}")
print(f"  cos(θ) = (v·w) / (||v|| ||w||) = {cos_theta:.4f}")
print(f"  θ = {theta_deg:.2f}°")

In [ ]:
# 3. 노름 (Norm) - 벡터의 길이/크기

In [ ]:
print("\n3. 다양한 노름 (벡터의 '크기' 측정)")
print("-" * 45)

x = np.array([3.0, -4.0, 0.0, 2.0])
print(f"x = {x}")

# L1 노름: 맨해튼 거리
l1_norm = np.linalg.norm(x, ord=1)
print(f"\nL1 노름 (||x||₁): {l1_norm}")
print(f"  = |3| + |-4| + |0| + |2| = {abs(3) + abs(-4) + abs(0) + abs(2)}")
print(f"  용도: 희소성 유도 (Lasso 정규화)")

# L2 노름: 유클리드 거리
l2_norm = np.linalg.norm(x, ord=2)
print(f"\nL2 노름 (||x||₂): {l2_norm}")
print(f"  = √(3² + 4² + 0² + 2²) = √{3**2 + 4**2 + 0**2 + 2**2} = {np.sqrt(29):.4f}")
print(f"  용도: 거리 측정, 가중치 정규화")

# 무한대 노름: 최댓값
linf_norm = np.linalg.norm(x, ord=np.inf)
print(f"\nL∞ 노름 (||x||∞): {linf_norm}")
print(f"  = max(|3|, |-4|, |0|, |2|) = 4")

In [ ]:
# 4. 코사인 유사도 (Cosine Similarity)

In [ ]:
print("\n4. 코사인 유사도 (토큰 유사성의 핵심)")
print("-" * 45)

# 트랜스포머에서 두 토큰 임베딩의 유사도
embed_cat = np.array([0.8, 0.3, 0.5, -0.1])
embed_dog = np.array([0.7, 0.4, 0.4, 0.0])
embed_car = np.array([-0.2, 0.9, -0.3, 0.8])

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_cat_dog = cosine_similarity(embed_cat, embed_dog)
sim_cat_car = cosine_similarity(embed_cat, embed_car)

print(f"임베딩 예시 (4차원):")
print(f"  cat: {embed_cat}")
print(f"  dog: {embed_dog}")
print(f"  car: {embed_car}")
print(f"\n코사인 유사도:")
print(f"  sim(cat, dog) = {sim_cat_dog:.4f}  ← 높음 (의미적 유사)")
print(f"  sim(cat, car) = {sim_cat_car:.4f}  ← 낮음 (의미적 다름)")

In [ ]:
# 5. 투영 (Projection)

In [ ]:
print("\n5. 벡터 투영 (어텐션의 기하학)")
print("-" * 45)

# v를 w 방향으로 투영
# proj_w(v) = (v·w / w·w) * w

def project(v, w):
    """v를 w 방향으로 투영"""
    return (np.dot(v, w) / np.dot(w, w)) * w

v = np.array([4.0, 3.0])
w = np.array([1.0, 0.0])  # x축 방향

proj = project(v, w)
orthogonal = v - proj  # 직교 성분

print(f"v = {v}")
print(f"w = {w} (x축 방향)")
print(f"proj_w(v) = {proj}  ← w 방향 성분")
print(f"v - proj = {orthogonal}  ← w에 직교하는 성분")
print(f"검증: proj · orthogonal = {np.dot(proj, orthogonal):.10f} ≈ 0 (직교)")

print("\n어텐션에서의 투영:")
print("  Q, K 벡터의 내적 = Q를 K 방향으로 투영 × ||K||")
print("  이것이 '유사도 점수'가 됩니다!")

In [ ]:
# 6. 고차원 벡터의 직관

In [ ]:
print("\n6. 고차원 벡터의 특이한 성질")
print("-" * 45)

d = 768  # GPT-2 small의 d_model
np.random.seed(42)

# 고차원에서 랜덤 벡터들은 거의 직교!
n_vectors = 5
random_vecs = [np.random.randn(d) for _ in range(n_vectors)]

print(f"차원: {d}, 랜덤 벡터 {n_vectors}개")
print("\n쌍별 코사인 유사도:")
for i in range(n_vectors):
        for j in range(i+1, n_vectors):
        sim = cosine_similarity(random_vecs[i], random_vecs[j])
        print(f"  vec{i} · vec{j} = {sim:.4f}")

print("\n→ 고차원에서 랜덤 벡터는 거의 직교합니다!")
print("  이것이 '중첩(superposition)'이 가능한 이유입니다.")
print("  각 벡터가 거의 독립적인 '방향'을 가질 수 있습니다.")